## Blocks World

This notebook models the classic **blocks world** domain with three blocks in the unified-planning library.

The world has a table, a set of blocks and a robotic arm (gripper). The arm can hold at most one block at a time and can only pick up a block that has nothing on top of it (a *clear* block). We use the standard four operators:

 - `pick_up(x)` – pick up block `x` from the table,
 - `put_down(x)` – put the held block `x` on the table,
 - `stack(x, y)` – put the held block `x` on top of block `y`,
 - `unstack(x, y)` – pick up block `x` from the top of block `y`.

As the problem instance we use the **Sussman anomaly**:

```
 Initial state        Goal state
                         A
   C                     B
   A   B                 C
 ---------           ---------
```

### Setup

In [1]:
#%pip install "unified-planning[pyperplan,fast-downward]"

In [2]:
from unified_planning.shortcuts import *

# Do not print the credits of the planning engines every time we call them
get_environment().credits_stream = None

### Domain

#### Types and fluents

There is a single user type, `Block`. The state of the world is described by the following boolean fluents:

 - `on(x, y)` – block `x` lies directly on block `y`,
 - `on_table(x)` – block `x` lies on the table,
 - `clear(x)` – there is nothing on top of block `x`,
 - `holding(x)` – the arm holds block `x`,
 - `hand_empty` – the arm holds nothing.

In [3]:
Block = UserType("Block")

on = Fluent("on", BoolType(), x=Block, y=Block)
on_table = Fluent("on_table", ...)
clear = Fluent("clear", ...)
holding = Fluent("holding", ...)
hand_empty = Fluent("hand_empty", ...)

#### Actions

Each action lists its preconditions and its effects. All fluents not mentioned in the effects keep their values (the frame assumption).

In [4]:
pick_up = InstantaneousAction("pick_up", x=Block)
x = pick_up.parameter("x")
pick_up.add_precondition(clear(x))
pick_up.add_precondition(on_table(x))
pick_up.add_precondition(hand_empty)
pick_up.add_effect(on_table(x), False)
pick_up.add_effect(clear(x), False)
pick_up.add_effect(hand_empty, False)
pick_up.add_effect(holding(x), True)

put_down = InstantaneousAction("put_down", x=Block)
# add precoditions and effects

stack = InstantaneousAction("stack", x=Block, y=Block)
# add precoditions and effects

unstack = InstantaneousAction("unstack", x=Block, y=Block)
# add precoditions and effects

print(stack)

action stack(Block x, Block y) {
    preconditions = [
      holding(x)
      clear(y)
      (not (x == y))
    ]
    effects = [
      holding(x) := false
      clear(y) := false
      on(x, y) := true
      clear(x) := true
      hand_empty := true
    ]
  }


### Problem: the Sussman anomaly

We create three blocks and describe the initial state and the goal. Fluents not set explicitly default to `False`.

In [5]:
problem = Problem("blocks_world")
for f in (on, on_table, ...): # add fluents
    problem.add_fluent(f, default_initial_value=False)
problem.add_actions([...])

A, B, C = (Object(name, Block) for name in "ABC")
problem.add_objects([A, B, C])

# Initial state: C is on A, A and B are on the table, the arm is empty
problem.set_initial_value(on(C, A), True)
# add other initial values


# Goal: the tower A on B on C
problem.add_goal(on(A, B))
problem.add_goal(on(B, C))

print(problem)

problem name = blocks_world

types = [Block]

fluents = [
  bool on[x=Block, y=Block]
  bool on_table[x=Block]
  bool clear[x=Block]
  bool holding[x=Block]
  bool hand_empty
]

actions = [
  action pick_up(Block x) {
    preconditions = [
      clear(x)
      on_table(x)
      hand_empty
    ]
    effects = [
      on_table(x) := false
      clear(x) := false
      hand_empty := false
      holding(x) := true
    ]
  }
  action put_down(Block x) {
    preconditions = [
      holding(x)
    ]
    effects = [
      holding(x) := false
      on_table(x) := true
      clear(x) := true
      hand_empty := true
    ]
  }
  action stack(Block x, Block y) {
    preconditions = [
      holding(x)
      clear(y)
      (not (x == y))
    ]
    effects = [
      holding(x) := false
      clear(y) := false
      on(x, y) := true
      clear(x) := true
      hand_empty := true
    ]
  }
  action unstack(Block x, Block y) {
    preconditions = [
      on(x, y)
      clear(x)
      hand_empty
    ]
 

### Solving the problem

We let the library pick a suitable planner for this problem kind with `OneshotPlanner(problem_kind=...)`.

In [6]:
with OneshotPlanner(problem_kind=problem.kind) as planner:
    result = planner.solve(problem)
    print(f"Planner: {planner.name}, status: {result.status.name}")
    plan = result.plan
    print(plan)

Planner: Fast Downward, status: SOLVED_SATISFICING
SequentialPlan:
    unstack(C, A)
    put_down(C)
    pick_up(B)
    stack(B, C)
    pick_up(A)
    stack(A, B)


To get a plan with the minimum number of actions, we add a quality metric (minimize the plan length) and require an optimality guarantee from the planner. The anomaly needs 6 actions: C has to be moved to the table first, before B can be put on C and A on B.

In [ ]:
problem.add_quality_metric(MinimizeSequentialPlanLength())

with OneshotPlanner(
    problem_kind=problem.kind,
    optimality_guarantee=OptimalityGuarantee.SOLVED_OPTIMALLY,
) as planner:
    result = planner.solve(problem)
    print(f"Planner: {planner.name}, status: {result.status.name}")
    plan = result.plan
    print(plan)

### Simulating the plan

We use the `SequentialSimulator` to replay the plan step by step and draw the towers in each intermediate state.

In [ ]:
blocks = [A, B, C]

def draw(state):
    """Return an ASCII drawing of the towers in a given state."""
    def value(fluent):
        return state.get_value(fluent).bool_constant_value()

    towers = []
    for base in blocks:
        if value(on_table(base)):
            tower = [base]
            while True:
                above = [b for b in blocks if value(on(b, tower[-1]))]
                if not above:
                    break
                tower.append(above[0])
            towers.append(tower)
    held = [b.name for b in blocks if value(holding(b))]
    height = max(len(t) for t in towers) if towers else 0
    lines = [f"  hand: {held[0] if held else '-'}"]
    for level in reversed(range(height)):
        lines.append("  " + " ".join(t[level].name if level < len(t) else " " for t in towers))
    lines.append("  " + "-" * (2 * len(towers) - 1 if towers else 1))
    return "\n".join(lines)

with SequentialSimulator(problem) as simulator:
    state = simulator.get_initial_state()
    print("initial state")
    print(draw(state))
    for step, action in enumerate(plan.actions, start=1):
        state = simulator.apply(state, action)
        print(f"\n{step}. {action}")
        print(draw(state))
    print("\ngoal reached:", simulator.is_goal(state))

### Exporting to PDDL

The same problem can be exported to PDDL, e.g. to be solved by any external PDDL planner.

In [ ]:
from unified_planning.io import PDDLWriter

writer = PDDLWriter(problem)
print(writer.get_domain())
print(writer.get_problem())